In [1]:
import os
import csv
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import traceback
import math
import sys
import time
import socket
import time
from time import sleep
import pyvisa
from hardware.hmp4040.hmp4040 import hmp4040
from hardware.msox2024a.msox2024a import msox2024a

### Server

In [2]:
# Set server address to machines IP
SERVER_ADDR = "0.0.0.0"

# These constants should match the client
BUF_SIZE = 128
TEST_ITERATIONS = 999
SERVER_PORT = 4242

# Open socket to the server
sock = socket.socket()
sock.bind((SERVER_ADDR, SERVER_PORT))
sock.listen(1)
print("server listening on", SERVER_ADDR, SERVER_PORT)

server listening on 0.0.0.0 4242


In [3]:
def hvdc_cmd(cmd = None):

    sock.settimeout(5)   # <<< FIX #1 — timeout for accept()

    try:
        con, addr = sock.accept()
    except socket.timeout:
        print("No client connected after 5 seconds")
        return

    print("Client connected from", addr)
    
    message = None
    if cmd == None:
        message = input(">> ").encode('utf-8')
    else:
        message = cmd.encode('utf-8')
    try:
        if message.decode() == "DISCONNECT":
            exit()
    
        # Send message with end byte 0x04
        message_len = con.send(message + b'\x04')
        print(f'Written {message_len} bytes to client')
    
        # Read response from client
        read_buf = b''
        end_byte = b'\x04'
        con.settimeout(5)  # 5 second timeout
        while True:
            try:
                buf = con.recv(64)  # read in chunks
                if not buf:
                    break
                read_buf += buf
                if end_byte in buf:
                    # stop once end byte is detected
                    break
            except socket.timeout:
                print("Timed out waiting for client response")
                break
        
        # Remove everything after the end byte
        if end_byte in read_buf:
            read_buf = read_buf.split(end_byte)[0]
        
        print(f'Read {len(read_buf)} bytes from client')
        if read_buf:
            print("Ack:", read_buf.decode())

    except Exception as e:
        print(e)

### Supplies

In [ ]:
rm = pyvisa.ResourceManager()
hmp4040_port = input("PSU Port: ")
psu = rm.open_resource(f'ASRL{int(hmp4040_port)}::INSTR')

hmp4040 = hmp4040(pyvisa_instr=psu)
print("Connected to", psu.query("*IDN?"))

In [ ]:
#rm = pyvisa.ResourceManager() 
msox2024a_port = input("Scope Address: ")
scope = msox2024a(rm, msox2024a_port)
scope.Connect()
#     USB0::0x0957::0x1796::MY62160147::0::INSTR
print("Connected to", scope.query("*IDN?"))

### Suppy Defs

In [ ]:
def set_voltage(chan, volts, unmute=False):
    """
    Chan: channel number
    Volts: voltage (float)
    unmute: repeats channel and set voltage if True, otherwise does not if left default
    """
    psu.write(f"INST OUT{int(chan)}")
    psu.write(f"VOLT {float(volts)}")
    if unmute==True:
        print(f"SetV: {psu.query('VOLT?')}  Chan: {psu.query('INST:NSEL?')}")

In [ ]:
def set_current(chan, current, unmute=False):
    """
    Chan: channel number
    Current: current (float)
    unmute: repeats channel and set current if True, otherwise does not if left default
    """
    psu.write(f"INST OUT{int(chan)}")
    psu.write(f"CURR {float(current)}")
    if unmute==True:
        print(f"SetI: {psu.query('CURR?')}  Chan: {psu.query('INST:NSEL?')}")

In [ ]:
def set(chan, volts, current, unmute=False):
    """
    Chan: channel number
    Volts: voltage (float)
    Current: current (float)
    unmute: repeats channel and set voltage + current if True, otherwise does not if left default
    """
    psu.write(f"INST OUT{int(chan)}")
    psu.write(f"APPLY {float(volts)},{float(current)}")
    if unmute==True:
        print(f"SetV: {psu.query('VOLT?')} SetI: {psu.query('CURR?')}  Chan: {psu.query('INST:NSEL?')}")

In [ ]:
def output(chan, state, unmute=False):
    """
    Chan: channel number
    State: 0/1/False/True turns on or off output
    unmute: repeats channel and state of output
    IF A CHANNEL IS TURNED ON THEN TURNED OFF, THE OUTPUT WILL REMAIN ON BUT CHANNEL WILL BE OFF
    """
    if state == 1 or state == True:
        psu.write(f"OUTP ON")
    elif state == 0 or state == False:
        psu.write(f"OUTP OFF")
    if unmute==True:
        print(f"SetI: {psu.query('OUTP:STAT?')}  Chan: {psu.query('INST:NSEL?')}")

In [ ]:
def output_all(state):
    """
    State: 0/1/False/True turns on or off all outputs
    """
    if state == 1 or state == True:
        psu.write(f"OUTP:GEN ON")
    elif state == 0 or state == False:
        psu.write(f"OUTP:GEN OFF")

In [ ]:
def autoscale():
    """
    AUTOSCALE
    """
    scope.write(":AUT");

In [ ]:
def display(chan, state):
    """
    Chan: channel number
    State: 0/False/1/True turns on or off display
    """
    if state == 0 or state == False:
        scope.write(f":CHAN{chan}:DISP 0")
    elif state == 1 or state == True:
        scope.write(f":CHAN{chan}:DISP 1")

In [ ]:
def run():
    scope.write(":RUN")

In [ ]:
def stop():
    scope.write(":STOP")

In [ ]:
def set_scale(chan, scale, suffix):
    """
    Chan: channel number
    Scale: vertical scale
    Suffix: "V" or "mV"
    """
    scope.write(f":CHAN{chan}:SCAL {scale}{suffix}")

In [ ]:
def set_window(window):
    """
    Window: set "MAIN" for main window measuremnt, "ZOOM" for zoom window measurement, "AUTO" does zoom when possible, else uses main window
    """
    scope.write(f':MEAS:WIND {window}')

In [ ]:
def get_voltage(chan):
    """
    Chan: channel number
    Returns the top voltage (average top value, not necessarily peak value)
    """
    return float(scope.query(f":MEAS:VTOP? CHAN{chan}"))

In [ ]:
def get_vbase(chan):
    """
    Chan: channel number
    Returns the top voltage (average top value, not necessarily peak value)
    """
    return float(scope.query(f":MEAS:VBAS? CHAN{chan}"))

In [ ]:
def get_vamplitude(chan):
    """
    Chan: channel number
    Returns the voltage amplitude (top value - base value)
    """
    return float(scope.query(f":MEAS:VAMP? CHAN{chan}"))

In [ ]:
def get_vrms(chan):
    """
    Chan: channel number
    Returns RMS
    """
    return float(scope.query(f":MEAS:VRMS? CHAN{chan}"))

In [ ]:
def get_vaverage(chan, mode):
    """
    Chan: channel number
    Mode: "CYCLe" or "DISPlay", respectively measured over 3+ periods or the scope screen
    Returns 
    """
    return float(scope.query(f":MEAS:VAV? {mode},CHAN{chan}"))

In [ ]:
def get_vmax(chan):
    """
    Chan: channel number
    Returns maximum voltage
    """
    return float(scope.query(f":MEAS:VMAX? CHAN{chan}"))

In [ ]:
def get_vmin(chan):
    """
    Chan: channel number
    Returns minimum voltage
    """
    return float(scope.query(f":MEAS:VMIN? CHAN{chan}"))

In [ ]:
def get_vpp(chan):
    """
    Chan: channel number
    Returns peak to peak voltage (vmax - vmin)
    """
    return float(scope.query(f":MEAS:VPP? CHAN{chan}"))

In [ ]:
def get_frequency(chan):
    """
    Chan: channel number
    Returns frequency over a full screen
    """
    return float(scope.query(f":MEAS:FREQ? CHAN{chan}"))

In [ ]:
def get_all():
    """
    Returns all measurents on scope
    """
    scope.write(':MEAS:ALL')

## ACTUAL CODE STARTS HERE

#### Maximum resolution of frequency --> freq / Fclk, pwm --> Fclk / freq - 1


In [ ]:
freq = np.linspace(0.9e6, 1.15e6, 51)
print(freq)

In [ ]:
pwm = np.linspace(0.2, 0.8, 31)
print(pwm)

In [ ]:
filename = "3VIN_4TURN_LUT_sweep_data_" + input("datetime") + ".csv"
if not os.path.exists(filename):
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        write_to_csv = ["kOhm", "kHz", "IIN", "PIN", "IRMS", "VRMS", "POUT", "EFF", "ZVS"]
        writer.writerow(write_to_csv)

In [ ]:
curr_time = time.time()

for l in loads:
    for f in freq:
        for p in pwm:
            try:
                set(1, 3, 20)
                output(1, True)
                
                hvdc_cmd("PINFREQDUTY " + "16 "+ str(freq) + " " + str(pwm))
                hvdc_cmd("EN 1")
                
            except Exception as e:
                IIN = 1e-9
                PIN = 1e-9
                POUT = 1e-9
                EFF = 1e-9
                hvdc_cmd("EN 0")
                output(1, 0)

            sleep_ms(100)
            VOUT = get_voltage(1)
            IOUT = get_voltage(2) / ROUTSHUNT
            IIN = (get_voltage(3) - get_voltage(4)) / RINSHUNT
            PIN = 3 * IIN - 3 * RWIRE
            POUT = VOUT * IOUT
            EFF = POUT / PIN
                
    
            elapsed = int(time.time() - curr_time)

            hvdc_cmd("EN 0")
            output(1, 0)
    
            print(
                f"\rElapsed: {elapsed} F: {f/1e3:.1f} kHz, R: {l/1e3:.1f}  kOhm, PWM: {p}, IIN: {IIN} PIN: {PIN} POUT: {POUT} EFF: {EFF}",
                end="",
                flush=True
            )
            
            with open(filename, "a", newline="") as csvfile:
                writer = csv.writer(csvfile)
                write_to_csv = [l, f, IIN, PIN, IRMS, VRMS, POUT, EFF, ZVS]
                writer.writerow(write_to_csv)

            sleep_ms(300)

print("LUT Finished")

### PLOTTING

## TESTING

In [ ]:
hvdc_cmd("CLK " + str(150000))

In [ ]:
hvdc_cmd("PINFREQDUTY " + str(16) + " " + str(int(25e3)) + " " + str(50))

In [ ]:
hvdc_cmd("PINFREQDUTY " + str(15) + " " + str(int(1e6)) + " " + str(50))

In [ ]:
hvdc_cmd("PINFREQDUTY " + str(14) + " " + str(int(1e6)) + " " + str(50))

In [ ]:
hvdc_cmd("EN 1")

In [ ]:
hvdc_cmd("EN 0")

In [ ]:
# Wait for the client
con = None
con, addr = sock.accept()
con.settimeout(15)
print("client connected from", addr)

# Repeat test for a number of iterations
for test_iteration in range(TEST_ITERATIONS):

    # Generate a buffer of random data
    # random_list = []
    # for n in range(BUF_SIZE):
    #     random_list.append(random.randint(0, 255))
    # write_buf = bytearray(random_list)

    message = input()
    write_buf = message.encode("ascii")
    write_buf = write_buf.ljust(BUF_SIZE, b'\x00')

    # Write BUF_SIZE bytes to the client
    write_len = con.send(bytearray(write_buf))
    print('Written %d bytes to client' % write_len, end="", flush=True)

    # Check size of data written
    if write_len != BUF_SIZE:
        raise RuntimeError('wrong amount of data written %d' % write_len)

    # Read the data back from the client
    total_size = BUF_SIZE
    read_buf = b''
    while total_size > 0:
        try:
            buf = con.recv(BUF_SIZE)
        except:
            print("could not receive message")
        print('read %d bytes from client' % len(buf), end="", flush=True)
        total_size -= len(buf)
        read_buf += buf

    result = read_buf.rstrip(b'\x00').decode("ascii")

    if result == 'exit':
        break

    print("read_buf (hex):", result)
    # # Check size of data received
    # if len(read_buf) != BUF_SIZE:
    #     raise RuntimeError('wrong amount of data read')

    # # Check the data sent and received
    # if read_buf != write_buf:
    #     EN 14 0
    #raise RuntimeError('buffer mismatch')

# All done
con.close()
sock.close()
print("test completed")

client connected from ('192.168.8.234', 64279)


 freq-pwm 640000 50 15


Written 128 bytes to clientread 128 bytes from clientread_buf (hex): freq-pwm 640000 50 15                                                                                                         OK


 freq-pwm 640000 50 14


Written 128 bytes to clientread 128 bytes from clientread_buf (hex): freq-pwm 640000 50 14                                                                                                         OK


 EN 14 1


Written 128 bytes to clientread 128 bytes from clientread_buf (hex): EN 14 1                                                                                                                       OK


Written 128 bytes to clientcould not receive message
read_buf (hex): EN 14 1                                                                                                                       OK


Written 128 bytes to clientcould not receive message
read_buf (hex): EN 14 1                                                                                                                       OK
